# ChronoPDE V2 Phase 6B — one-shot confirmatory evaluation

Attach the private Phase 5 results and Phase 6 confirmatory-data datasets, enable Internet, and select GPU T4 x2. This performs no training or checkpoint selection.

In [ ]:
import os
import shutil
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import torch

URL = "https://github.com/madhavkapoor13/ChronoPDE.git"
BRANCH = "codex/chronopde-v2-phase6"
REPOSITORY = Path("/kaggle/working/ChronoPDE")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPOSITORY)], check=True
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPOSITORY)], check=True
)
phase5 = list(Path("/kaggle/input").rglob("chronopde_v2_phase5_outputs.zip"))
data = list(Path("/kaggle/input").rglob("chronopde_v2_confirmatory.h5"))
assert len(phase5) == len(data) == 1, (phase5, data)
PHASE5, DATA = phase5[0], data[0]
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator."
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

In [ ]:
def run_one(model, seed, gpu):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    command = [
        sys.executable,
        "scripts/chronopde_v2.py",
        "phase6",
        "evaluate",
        "--phase5-archive",
        str(PHASE5),
        "--data-path",
        str(DATA),
        "--model",
        model,
        "--seed",
        str(seed),
        "--device",
        "cuda",
        "--format",
        "json",
    ]
    process = subprocess.Popen(
        command,
        cwd=REPOSITORY,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(f"[{model} s{seed} gpu{gpu}] {line}", end="", flush=True)
    return model, seed, process.wait()


return_codes = {}
for seed in range(5):
    jobs = [("fft", seed, 0), ("dct", seed, 1 if GPU_COUNT >= 2 else 0)]
    if GPU_COUNT >= 2:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [executor.submit(run_one, *job) for job in jobs]
            for future in as_completed(futures):
                model, completed_seed, code = future.result()
                return_codes[(model, completed_seed)] = code
    else:
        for job in jobs:
            model, completed_seed, code = run_one(*job)
            return_codes[(model, completed_seed)] = code
print("Evaluation return codes:", return_codes)

In [ ]:
successful = len(return_codes) == 10 and all(code == 0 for code in return_codes.values())
if successful:
    command = [
        sys.executable,
        "scripts/chronopde_v2.py",
        "phase6",
        "collect",
        "--phase5-archive",
        str(PHASE5),
        "--data-path",
        str(DATA),
        "--format",
        "json",
    ]
    collected = subprocess.run(command, cwd=REPOSITORY)
    successful = collected.returncode == 0
if successful:
    packages = list(
        (REPOSITORY / "artifacts/chronopde_v2/runs").rglob(
            "chronopde_v2_phase6_confirmatory_results.zip"
        )
    )
    successful = len(packages) == 1
if successful:
    output = Path("/kaggle/working/chronopde_v2_phase6_confirmatory_results.zip")
    shutil.copy2(packages[0], output)
    print("Download:", output)
else:
    partial = shutil.make_archive(
        "/kaggle/working/chronopde_v2_phase6_partial",
        "zip",
        root_dir=REPOSITORY / "artifacts/chronopde_v2/runs",
    )
    print("Preserve partial output:", partial)
assert successful, f"Confirmatory evaluation incomplete: {return_codes}"